In [24]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from googletrans import Translator, LANGUAGES
from gtts import gTTS


$\textcolor{red}{Scraper - Functions}$

In [25]:
def scrape_page(driver):
    articles = driver.find_elements(By.CLASS_NAME, "blog")
    news_list = []
    for article in articles:
        title = "N/A"
        url = "N/A"
        try:
            title_link = article.find_element(By.CLASS_NAME, "card-title").find_element(By.TAG_NAME, "a")
            title = title_link.text
            url = title_link.get_attribute("href")
        except:
            print("Couldn't get title or URL for an article")
        
        try:
            image_box = article.find_element(By.CLASS_NAME, "card-img-top")
            image = image_box.find_element(By.TAG_NAME, "img")
            image_url = image.get_attribute("data-srcset").split(",")[0].split()[0]
        except:
            print("No image found for " + title)
            image_url = "No image available"
        
        news_list.append([title, url, image_url])
    return news_list


 
$\textcolor{green}{Extract- Article -Content}$

In [26]:
def get_article_content(driver, url):
    driver.get(url)
    time.sleep(2)
    content = "No content available"
    try:
        content_box = driver.find_element(By.ID, "content-details")
        content = content_box.text
    except:
        try:
            content_box = driver.find_element(By.TAG_NAME, "article")
            content = content_box.text
        except:
            try:
                content_box = driver.find_element(By.CLASS_NAME, "content")
                content = content_box.text
            except:
                print("Couldn't find content for " + url)
    return content


$\textcolor{red}{Scroll- and- Load -More -Content}$ 

In [27]:
def scroll_and_load(driver):
    last_height = driver.execute_script("return document.body.scrollHeight")
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height


$\textcolor{red}{Save- Data- to -CSV}$

In [28]:
def save_to_csv(data, filename):
    df = pd.DataFrame(data, columns=["Title", "URL", "Image URL", "Content", "Translated Content"])
    df.to_csv(filename, index=False, encoding="utf-8")
    print("Saved data to " + filename)


$\textcolor{red}{Translate- Text}$  

In [29]:
def translate_text(text, target_language):
    translator = Translator()
    try:
        translated = translator.translate(text, src="am", dest=target_language).text
        return translated
    except Exception as e:
        print("Translation error: " + str(e))
        return text 


 $\textcolor{red}{Text-to-Speech}$

In [30]:
def text_to_speech(text, filename, language="en"):
    try:
        tts = gTTS(text=text, lang=language, slow=False)
        tts.save(filename)
        print("Saved audio file as " + filename)
    except Exception as e:
        print("Text-to-speech error: " + str(e))


 $\textcolor{red}{Start- Web- Scraper}$

In [33]:
print("Starting the web scraper")

print("Available languages: " + ", ".join(LANGUAGES.keys()))
target_language = input("Which language do you want to translate the Amharic content to? ").strip().lower()
if target_language not in LANGUAGES:
    print(target_language + " is not valid. Using 'en' (English) as default.")
    target_language = "en"

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

main_url = "https://am.al-ain.com"
driver.get(main_url)
print("Loading main page")

scroll_and_load(driver)


Starting the web scraper
Available languages: af, sq, am, ar, hy, az, eu, be, bn, bs, bg, ca, ceb, ny, zh-cn, zh-tw, co, hr, cs, da, nl, en, eo, et, tl, fi, fr, fy, gl, ka, de, el, gu, ht, ha, haw, iw, he, hi, hmn, hu, is, ig, id, ga, it, ja, jw, kn, kk, km, ko, ku, ky, lo, la, lv, lt, lb, mk, mg, ms, ml, mt, mi, mr, mn, my, ne, no, or, ps, fa, pl, pt, pa, ro, ru, sm, gd, sr, st, sn, sd, si, sk, sl, so, es, su, sw, sv, tg, ta, te, th, tr, uk, ur, ug, uz, vi, cy, xh, yi, yo, zu


Which language do you want to translate the Amharic content to?  fr


Loading main page


$\textcolor{red}{Scrape -Main- Page- Articles}$  

In [21]:
news_data = scrape_page(driver)
print("Found " + str(len(news_data)) + " articles on the main page!")

for i in range(len(news_data)):
    news = news_data[i]
    print("Getting content for " + news[0])
    content = get_article_content(driver, news[1])
    translated_content = translate_text(content, target_language)
    news.append(content)
    news.append(translated_content)
    
    audio_filename = "article_" + str(i + 1) + "_" + target_language + ".mp3"
    text_to_speech(translated_content, audio_filename, language=target_language)

save_to_csv(news_data, "main_news_data.csv")


No image found for ትራምፕ ከቀድሞው የአሜሪካ ፕሬዝደንት ጆን ኤፍ. ኬነዲ ግድያ ጋር የተያያዙ ሰነዶችን ይፋ አደረጉ
No image found for እስራኤል ከሁለት ወራት ተኩስ አቁም በኋላ በጋዛ ድጋሚ ሙሉ ጦርነት ማወጇን አስታወቀች
No image found for ጾታቸውን ያስቀየሩ ዜጎች የአሜሪካ ጦርን እንዳይቀላቀሉ በሚል ተላልፎ የነበረው ውሳኔ በፍርድ ቤት ተሻረ
No image found for ትራምፕ ከቀድሞው የአሜሪካ ፕሬዝደንት ጆን ኤፍ. ኬነዲ ግድያ ጋር የተያያዙ ሰነዶችን ይፋ አደረጉ
No image found for እስራኤል ከሁለት ወራት ተኩስ አቁም በኋላ በጋዛ ድጋሚ ሙሉ ጦርነት ማወጇን አስታወቀች
No image found for ጾታቸውን ያስቀየሩ ዜጎች የአሜሪካ ጦርን እንዳይቀላቀሉ በሚል ተላልፎ የነበረው ውሳኔ በፍርድ ቤት ተሻረ
No image found for ቱርክ የፕሬዝደንት ኢርዶጋንን ዋና ተቀናቃኝ አሰረች
No image found for ኤአይ በፈጠራ ስራ ላይ የሚያሳድረው አሉታዊ ተጽዕኖ
No image found for የአንድን ሀገር ወይም ግለሰብ ዲፕሎማሲያዊ ተጽዕኖ የሚወስኑ ጉዳዮች ምንድን ናቸው?
No image found for አርቴፊሻል ኢንተሊጀንስ የፈጠራ ስራን በምን መልኩ ይረዳል?
No image found for የመሬት መጠን መቀነስን የሚያስከትሉ መንስኤዎች
No image found for በካንሰር በሽታ ዙሪያ የተገኙ አዳዲስ የምርምር ውጤቶች
No image found for ህይወትን ቀለል አድርጎ መምራት ረጅም ዕድሜ ለመኖር ያለው ጠቀሜታ
Found 55 articles on the main page!
Getting content for ትራምፕ ከቀድሞው የአሜሪካ ፕሬዝደንት ጆን ኤፍ. ኬነዲ ግድያ ጋር የተያያዙ ሰነዶችን ይፋ አደረጉ
Saved

$\textcolor{red}{Scrape-Subpages}$

In [34]:
subpage_links = []
try:
    nav_bar = driver.find_element(By.CLASS_NAME, "navbar")
    links = nav_bar.find_elements(By.TAG_NAME, "a")
except:
    print("Couldn't find the navigation bar ")
    links = []

for link in links:
    href = link.get_attribute("href")
    if href and href.startswith("https://am.al-ain.com"):
        subpage_links.append(href)

print("Found " + str(len(subpage_links)) + " subpages to scrape!")

for subpage in subpage_links:
    print("Scraping subpage: " + subpage)
    driver.get(subpage)
    time.sleep(2)
    scroll_and_load(driver)
    
    subpage_data = scrape_page(driver)
    
    for i in range(len(subpage_data)):
        news = subpage_data[i]
        print("Getting content for " + news[0])
        content = get_article_content(driver, news[1])
        translated_content = translate_text(content, target_language)
        news.append(content)
        news.append(translated_content)
        
        page_name = subpage.split("/")[-1]
        if not page_name:
            page_name = subpage.split("/")[-2]
        audio_filename = page_name + "_article_" + str(i + 1) + "_" + target_language + ".mp3"
        text_to_speech(translated_content, audio_filename, language=target_language)
    
    page_name = subpage.split("/")[-1]
    if not page_name:
        page_name = subpage.split("/")[-2]
    save_to_csv(subpage_data, page_name + "_news_data.csv")


Found 13 subpages to scrape!
Scraping subpage: https://am.al-ain.com/
No image found for ትራምፕ ከቀድሞው የአሜሪካ ፕሬዝደንት ጆን ኤፍ. ኬነዲ ግድያ ጋር የተያያዙ ሰነዶችን ይፋ አደረጉ
No image found for እስራኤል ከሁለት ወራት ተኩስ አቁም በኋላ በጋዛ ድጋሚ ሙሉ ጦርነት ማወጇን አስታወቀች
No image found for ጾታቸውን ያስቀየሩ ዜጎች የአሜሪካ ጦርን እንዳይቀላቀሉ በሚል ተላልፎ የነበረው ውሳኔ በፍርድ ቤት ተሻረ
No image found for ትራምፕ ከቀድሞው የአሜሪካ ፕሬዝደንት ጆን ኤፍ. ኬነዲ ግድያ ጋር የተያያዙ ሰነዶችን ይፋ አደረጉ
No image found for እስራኤል ከሁለት ወራት ተኩስ አቁም በኋላ በጋዛ ድጋሚ ሙሉ ጦርነት ማወጇን አስታወቀች
No image found for ጾታቸውን ያስቀየሩ ዜጎች የአሜሪካ ጦርን እንዳይቀላቀሉ በሚል ተላልፎ የነበረው ውሳኔ በፍርድ ቤት ተሻረ
No image found for ቱርክ የፕሬዝደንት ኢርዶጋንን ዋና ተቀናቃኝ አሰረች
No image found for ኤአይ በፈጠራ ስራ ላይ የሚያሳድረው አሉታዊ ተጽዕኖ
No image found for የአንድን ሀገር ወይም ግለሰብ ዲፕሎማሲያዊ ተጽዕኖ የሚወስኑ ጉዳዮች ምንድን ናቸው?
No image found for አርቴፊሻል ኢንተሊጀንስ የፈጠራ ስራን በምን መልኩ ይረዳል?
No image found for የመሬት መጠን መቀነስን የሚያስከትሉ መንስኤዎች
No image found for በካንሰር በሽታ ዙሪያ የተገኙ አዳዲስ የምርምር ውጤቶች
No image found for ህይወትን ቀለል አድርጎ መምራት ረጅም ዕድሜ ለመኖር ያለው ጠቀሜታ
Getting content for ትራምፕ ከቀድሞው የአሜሪካ ፕሬዝደንት ጆን ኤፍ. ኬነ

WebDriverException: Message: unknown error: net::ERR_NAME_NOT_RESOLVED
  (Session info: chrome=134.0.6998.178)
Stacktrace:
	GetHandleVerifier [0x00F2C7F3+24435]
	(No symbol) [0x00EB2074]
	(No symbol) [0x00D806E3]
	(No symbol) [0x00D7DB6E]
	(No symbol) [0x00D71458]
	(No symbol) [0x00D72D60]
	(No symbol) [0x00D716E7]
	(No symbol) [0x00D71243]
	(No symbol) [0x00D70F51]
	(No symbol) [0x00D6EE30]
	(No symbol) [0x00D6F8BB]
	(No symbol) [0x00D8419E]
	(No symbol) [0x00E0FEE7]
	(No symbol) [0x00DED7BC]
	(No symbol) [0x00E0F20A]
	(No symbol) [0x00DED5B6]
	(No symbol) [0x00DBC54F]
	(No symbol) [0x00DBD894]
	GetHandleVerifier [0x012370A3+3213347]
	GetHandleVerifier [0x0124B0C9+3295305]
	GetHandleVerifier [0x0124558C+3271948]
	GetHandleVerifier [0x00FC7360+658144]
	(No symbol) [0x00EBB27D]
	(No symbol) [0x00EB8208]
	(No symbol) [0x00EB83A9]
	(No symbol) [0x00EAAAC0]
	BaseThreadInitThunk [0x761F7BA9+25]
	RtlInitializeExceptionChain [0x77E6C0CB+107]
	RtlClearBits [0x77E6C04F+191]


$\textcolor{red}{Close- Web- Driver}$ 

In [ ]:
driver.quit()
print("All done!")
